In [1]:
import os
import ast
import csv
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss, classification_report
from tqdm import tqdm

In [2]:
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

torch.backends.cudnn.benchmark = True   # ← faster convs, fixed input size

SR          = 16000
N_MELS      = 64
TIME_FRAMES = 128
NUM_CLASSES = 8
BATCH_SIZE  = 64      # ← 8 → 64 (16GB VRAM easily fits this small CRNN)
EPOCHS      = 40
LR          = 3e-4    # ← 1e-4 → 3e-4 (scaled up for bigger batch)

CLASS_NAMES = ["Scream", "Shout", "Crying", "Explosion", "Gunshot", "Glass", "Siren", "Alarm"]

BASE_PATH = os.path.join("fsd50k", "FSD50K.dev_audio_16k")   # ← relative, matches archive/fsd50k/...
CSV_PATH  = "final_meta.csv"                                  # ← sits next to notebook in archive/


Using: cuda


In [3]:
meta = pd.read_csv(CSV_PATH)
meta.columns = meta.columns.str.strip()
meta["fname"] = meta["fname"].astype(str) + ".wav"
print(meta.shape)
print(meta.head())

(12568, 6)
        fname                                             labels  \
0  161290.wav  Strum,Bowed_string_instrument,Musical_instrume...   
1  113364.wav                     Crying_and_sobbing,Human_voice   
2  155570.wav  Computer_keyboard,Typing,Domestic_sounds_and_h...   
3  368352.wav                                      Shatter,Glass   
4  208938.wav  Dishes_and_pots_and_pans,Domestic_sounds_and_h...   

                                                mids  split  \
0  /m/07s0s5r,/m/0l14_3,/m/04szw,/m/04rlf,/m/0342...    val   
1                                /m/0463cq4,/m/09l8g  train   
2                      /m/01m2v,/m/0316dw,/t/dd00071  train   
3                                /m/07rn7sz,/m/039jq  train   
4                               /m/04brg2,/t/dd00071  train   

                multi_label             primary_label  
0  [0, 0, 0, 0, 0, 0, 0, 0]                     Strum  
1  [0, 0, 1, 0, 0, 0, 0, 0]                       NaN  
2  [0, 0, 0, 0, 0, 0, 0, 0]      

In [4]:
meta["ml_parsed"] = meta["multi_label"].apply(ast.literal_eval)
arr = np.array(meta["ml_parsed"].tolist())

neg_counts      = (arr == 0).sum(axis=0)
pos_counts      = (arr == 1).sum(axis=0)
pos_counts_safe = np.where(pos_counts == 0, 1, pos_counts)
pos_weight_vals = neg_counts / pos_counts_safe
# Clamp to max 20 — raw weights up to 162 cause loss spikes and kernel crashes
pos_weight_vals = np.clip(pos_weight_vals, 1.0, 20.0)
pos_weight      = torch.tensor(pos_weight_vals, dtype=torch.float32).to(DEVICE)

print("Class counts (pos):", dict(zip(CLASS_NAMES, pos_counts)))
print("pos_weight:        ", dict(zip(CLASS_NAMES, pos_weight_vals.round(1))))

Class counts (pos): {'Scream': np.int64(254), 'Shout': np.int64(216), 'Crying': np.int64(109), 'Explosion': np.int64(1122), 'Gunshot': np.int64(348), 'Glass': np.int64(974), 'Siren': np.int64(77), 'Alarm': np.int64(1280)}
pos_weight:         {'Scream': np.float64(20.0), 'Shout': np.float64(20.0), 'Crying': np.float64(20.0), 'Explosion': np.float64(10.2), 'Gunshot': np.float64(20.0), 'Glass': np.float64(11.9), 'Siren': np.float64(20.0), 'Alarm': np.float64(8.8)}


In [5]:
AUDIO_CACHE_DIR = "audio_cache_16k"   # ← raw decoded audio cached here, reused every epoch
os.makedirs(AUDIO_CACHE_DIR, exist_ok=True)

class FSDDataset(Dataset):
    def __init__(self, df, base_path, augment=False):
        self.df        = df.reset_index(drop=True)
        self.base_path = base_path
        self.augment   = augment          # True for train only

    def __len__(self):
        return len(self.df)

    def _load_audio(self, file_path, fname):
        # ← cache raw decoded audio: librosa.load (I/O + resample) is the
        #   real bottleneck, not the mel step. Cache once, reuse forever.
        cache_path = os.path.join(AUDIO_CACHE_DIR, fname.replace(".wav", ".npy"))
        if os.path.exists(cache_path):
            return np.load(cache_path)
        audio, _ = librosa.load(file_path, sr=SR)
        np.save(cache_path, audio)
        return audio

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        file_path = os.path.join(self.base_path, row["fname"])

        labels = row["multi_label"]
        if isinstance(labels, str):
            labels = ast.literal_eval(labels)
        labels = np.array(labels, dtype=np.float32)

        audio = self._load_audio(file_path, row["fname"])
        sr    = SR

        # ── AUGMENTATION (train only) ─────────────────────────────
        if self.augment:
            audio = audio.copy()   # ← don't mutate cached array in place
            # 1. Add Gaussian noise (SNR ~20dB)
            if np.random.rand() < 0.5:
                noise = np.random.randn(len(audio)) * 0.005
                audio = audio + noise

            # 2. Time stretch ±10%
            if np.random.rand() < 0.4:
                rate  = np.random.uniform(0.9, 1.1)
                audio = librosa.effects.time_stretch(audio, rate=rate)

            # 3. Pitch shift ±2 semitones
            if np.random.rand() < 0.3:
                steps = np.random.randint(-2, 3)
                audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=steps)

            # 4. Random gain ±6dB
            if np.random.rand() < 0.4:
                gain  = np.random.uniform(0.5, 1.5)
                audio = audio * gain
        # ─────────────────────────────────────────────────────────

        mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        if mel_db.shape[1] < TIME_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, TIME_FRAMES - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :TIME_FRAMES]

        mel_db = mel_db[np.newaxis, :, :]

        return (
            torch.tensor(mel_db, dtype=torch.float32),
            torch.tensor(labels,  dtype=torch.float32)
        )


In [6]:
train_meta, temp_meta = train_test_split(meta, test_size=0.3, random_state=42)
val_meta,  test_meta  = train_test_split(temp_meta, test_size=0.5, random_state=42)

print(f"Train: {len(train_meta)} | Val: {len(val_meta)} | Test: {len(test_meta)}")

NUM_WORKERS = 8   # ← 0 → 8 (parallel CPU decode, keeps GPU fed)

train_loader = DataLoader(
    FSDDataset(train_meta, BASE_PATH, augment=True),   # ← augmentation ON
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True, prefetch_factor=4
)
val_loader = DataLoader(
    FSDDataset(val_meta,   BASE_PATH, augment=False),  # ← augmentation OFF
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True, prefetch_factor=4
)
test_loader = DataLoader(
    FSDDataset(test_meta,  BASE_PATH, augment=False),  # ← augmentation OFF
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=True, prefetch_factor=4
)


Train: 8797 | Val: 1885 | Test: 1886


In [7]:
class CRNN(nn.Module):
    """
    CRNN with optional SED (Sound Event Detection) mode.
    
    sed_mode=False  → clip-level output (B, 8)       ← same as before
    sed_mode=True   → frame-level output (B, T, 8)   ← SED
    
    Training with weak (clip-level) labels uses MIL max-pooling.
    """
    def __init__(self, num_classes, sed_mode=False):
        super().__init__()
        self.sed_mode = sed_mode

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.lstm    = nn.LSTM(32 * 16, 64, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)           # ← increased from 0.2
        self.fc      = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)                          # (B, 32, 16, T//4)
        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous()  # (B, T, C, F)
        x = x.view(b, t, c * f)                 # (B, T, 512)
        x, _ = self.lstm(x)                      # (B, T, 128)
        x = self.dropout(x)

        if self.sed_mode:
            # ── SED: per-frame predictions (B, T, 8) ──
            return self.fc(x)                    # (B, T, 8)
        else:
            # ── Clip-level: mean pool over time (B, 8) ──
            x = x.mean(dim=1)                    # (B, 128)
            return self.fc(x)                    # (B, 8)


def mil_loss(frame_logits, clip_labels, criterion):
    """
    Multiple Instance Learning loss for SED with weak labels.
    Uses max-pool over time frames → clip-level prediction.
    frame_logits: (B, T, 8)
    clip_labels:  (B, 8)
    """
    clip_logits = frame_logits.max(dim=1).values  # (B, 8)
    return criterion(clip_logits, clip_labels)


def frames_to_events(frame_probs, threshold=0.5, hop_size=0.04):
    """
    Convert frame-level probabilities to time-stamped events.
    frame_probs: (T, 8) numpy array, values 0-1
    Returns list of dicts: {class, start_s, end_s, max_score}
    """
    events = []
    for j, cls in enumerate(CLASS_NAMES):
        active   = frame_probs[:, j] > threshold
        in_event = False
        start    = 0
        scores   = []
        for t, is_active in enumerate(active):
            if is_active and not in_event:
                start    = t * hop_size
                in_event = True
                scores   = [frame_probs[t, j]]
            elif is_active and in_event:
                scores.append(frame_probs[t, j])
            elif not is_active and in_event:
                events.append({
                    "class":     cls,
                    "start_s":   round(start, 2),
                    "end_s":     round(t * hop_size, 2),
                    "max_score": round(float(np.max(scores)), 3),
                })
                in_event = False
        if in_event:
            events.append({
                "class":     cls,
                "start_s":   round(start, 2),
                "end_s":     round(len(active) * hop_size, 2),
                "max_score": round(float(np.max(scores)), 3),
            })
    return events

In [8]:
SED_MODE = True   # ← Set True to enable SED (frame-level output)
                   #   Set False for clip-level (original behaviour)

model     = CRNN(NUM_CLASSES, sed_mode=SED_MODE).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)  # ← added weight_decay
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5, verbose=True
)
print(model)
print(f"SED mode: {SED_MODE}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

CRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (lstm): LSTM(512, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=8, bias=True)
)
SED mode: True
Trainable params: 301,864


/user/HS400/as07181/miniconda3/envs/reid_env/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [9]:
# ── Training loop ────────────────────────────────────────────────
best_val_f1   = 0.0   # ← early stopping removed, PATIENCE gone, runs full EPOCHS

scaler = torch.amp.GradScaler('cuda')

with open("training_log.csv", "w", newline="") as f:
    csv.writer(f).writerow(["epoch","train_loss","hamming_acc","micro_f1","macro_f1"])

for epoch in range(EPOCHS):

    # ── TRAIN ──────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Train]", leave=False)

    for X, y in train_pbar:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(X)
            if SED_MODE:
                # MIL: max-pool frame logits → clip loss
                loss = mil_loss(outputs, y, criterion)
            else:
                loss = criterion(outputs, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss /= len(train_loader)

    # ── VALIDATE ───────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Val]", leave=False)

    with torch.no_grad():
        for X, y in val_pbar:
            with torch.amp.autocast('cuda'):
                logits = model(X.to(DEVICE))
                if SED_MODE:
                    # Clip-level prediction for validation: max over frames
                    probs = torch.sigmoid(logits.max(dim=1).values).cpu().numpy()
                else:
                    probs = torch.sigmoid(logits).cpu().numpy()

            preds = (probs > 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(y.numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    micro_f1    = f1_score(all_labels, all_preds, average="micro",  zero_division=0)
    macro_f1    = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    hamming_acc = 1 - hamming_loss(all_labels, all_preds)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Hamming: {hamming_acc:.4f} | "
        f"Micro F1: {micro_f1:.4f} | "
        f"Macro F1: {macro_f1:.4f}"
    )

    scheduler.step(micro_f1)

    if micro_f1 > best_val_f1:
        best_val_f1 = micro_f1
        torch.save(model.state_dict(), "best_model_CRNN_fast.pth")
        print(f"  --> Best saved (Micro F1: {best_val_f1:.4f})")
    # ← early stopping removed: no break, always continues to next epoch

    with open("training_log.csv", "a", newline="") as f:
        csv.writer(f).writerow([epoch+1,round(train_loss,4),round(hamming_acc,4),round(micro_f1,4),round(macro_f1,4)])

Epoch 01/40 | Loss: 0.8451 | Hamming: 0.9137 | Micro F1: 0.3569 | Macro F1: 0.1918
  --> Best saved (Micro F1: 0.3569)


Epoch 02/40 | Loss: 0.6944 | Hamming: 0.9029 | Micro F1: 0.3765 | Macro F1: 0.2320
  --> Best saved (Micro F1: 0.3765)


Epoch 03/40 | Loss: 0.6332 | Hamming: 0.9123 | Micro F1: 0.4027 | Macro F1: 0.2747
  --> Best saved (Micro F1: 0.4027)


Epoch 04/40 | Loss: 0.5891 | Hamming: 0.9046 | Micro F1: 0.4110 | Macro F1: 0.3148
  --> Best saved (Micro F1: 0.4110)


Epoch 05/40 | Loss: 0.5607 | Hamming: 0.9090 | Micro F1: 0.4255 | Macro F1: 0.3556
  --> Best saved (Micro F1: 0.4255)


...

Epoch 36/40 | Loss: 0.3563 | Hamming: 0.9347 | Micro F1: 0.5211 | Macro F1: 0.4541


Epoch 37/40 | Loss: 0.3557 | Hamming: 0.9353 | Micro F1: 0.5244 | Macro F1: 0.4601


Epoch 38/40 | Loss: 0.3563 | Hamming: 0.9351 | Micro F1: 0.5220 | Macro F1: 0.4532


Epoch 39/40 | Loss: 0.3555 | Hamming: 0.9342 | Micro F1: 0.5214 | Macro F1: 0.4584


Epoch 40/40 | Loss: 0.3518 | Hamming: 0.9373 | Micro F1: 0.5301 | Macro F1: 0.4619


In [ ]:
# ── Load best model ──
model.load_state_dict(torch.load("best_model_CRNN_fast.pth", map_location=DEVICE, weights_only=True))
model.eval()
print("Best model loaded: best_model_CRNN_fast.pth")

# ── Collect val-set probabilities for threshold search ──
# Cell 10 needs probs_all and labels_all — we build them here
probs_all  = []
labels_all = []

with torch.no_grad():
    for X, y in tqdm(val_loader, desc="Collecting val probs"):
        with torch.amp.autocast("cuda"):
            probs = torch.sigmoid(model(X.to(DEVICE))).float().cpu().numpy()
        probs_all.append(probs)
        labels_all.append(y.numpy())

probs_all  = np.vstack(probs_all)
labels_all = np.vstack(labels_all)
print(f"Val probs collected: {probs_all.shape}")


In [ ]:
# ── Global threshold sweep ────────────────────────────────────────
# SED mode: probs_all is (N, T, 8) → max-pool to (N, 8) for clip-level eval
if probs_all.ndim == 3:
    probs_clip = probs_all.max(axis=1)   # (N, 8)
    print(f"SED mode: max-pooled {probs_all.shape} → {probs_clip.shape}")
else:
    probs_clip = probs_all               # already (N, 8)

best_micro_f1  = 0.0
best_threshold = 0.5

print(f"{'Threshold':>12} | {'Micro F1':>10} | {'Macro F1':>10} | {'Hamming Acc':>12}")
print("-" * 54)

for t in np.arange(0.10, 0.91, 0.05):
    preds = (probs_clip > t).astype(int)

    if preds.sum() == 0:
        print(f"{t:>12.2f} | (no predictions)")
        continue

    micro = f1_score(labels_all, preds, average="micro", zero_division=0)
    macro = f1_score(labels_all, preds, average="macro", zero_division=0)
    h_acc = 1 - hamming_loss(labels_all, preds)
    print(f"{t:>12.2f} | {micro:>10.4f} | {macro:>10.4f} | {h_acc:>12.4f}")

    if micro > best_micro_f1:
        best_micro_f1  = micro
        best_threshold = round(float(t), 2)

np.save("best_thresholds.npy", np.full(NUM_CLASSES, best_threshold))
print(f"\nBest threshold : {best_threshold:.2f}")
print(f"Best Micro F1  : {best_micro_f1:.4f}")

# ── Test set eval ─────────────────────────────────────────────────
test_probs, test_labels = [], []

with torch.no_grad():
    for X, y in tqdm(test_loader, desc="Test eval"):
        with torch.amp.autocast("cuda"):
            probs = torch.sigmoid(model(X.to(DEVICE))).float().cpu().numpy()
        test_probs.append(probs)
        test_labels.append(y.numpy())

test_probs  = np.vstack(test_probs)
test_labels = np.vstack(test_labels)

# Max-pool if SED
if test_probs.ndim == 3:
    test_probs = test_probs.max(axis=1)

test_preds = (test_probs > best_threshold).astype(int)

print("\n── Test Set Results ──")
print(classification_report(test_labels, test_preds,
      target_names=CLASS_NAMES, zero_division=0))

## Per-Class SED Threshold Tuning
Find optimal threshold per class — same approach as CLAP tuning.

In [ ]:
# ── PER-CLASS SED THRESHOLD TUNING ──────────────────────────────
# Find best threshold per class using val set probs
# Uses probs_clip (max-pooled from SED) already computed in cell 10

from sklearn.metrics import f1_score, average_precision_score
import numpy as np

thresholds = np.arange(0.10, 0.95, 0.01)
sed_best_thresholds = {}
CLASSES_LOCAL = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]

print(f"{'Class':12s} {'Best T':>7} {'F1 flat':>9} {'F1 tuned':>9} {'Gain':>7}")
print("-" * 50)

for j, cls in enumerate(CLASSES_LOCAL):
    gt_col    = labels_all[:, j]
    score_col = probs_clip[:, j]

    # F1 at best global threshold (0.75)
    f1_flat = f1_score(gt_col, (score_col > best_threshold).astype(int), zero_division=0)

    best_f1, best_t = 0.0, best_threshold
    for t in thresholds:
        pred = (score_col > t).astype(int)
        f1   = f1_score(gt_col, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    sed_best_thresholds[cls] = round(float(best_t), 2)
    gain = best_f1 - f1_flat
    print(f"{cls:12s} {best_t:7.2f} {f1_flat:9.3f} {best_f1:9.3f} {gain:+7.3f}")

print(f"\nSED best thresholds: {sed_best_thresholds}")

# Re-evaluate with per-class tuned thresholds
all_pseudo_tuned = np.zeros_like(probs_clip, dtype=np.float32)
for j, cls in enumerate(CLASSES_LOCAL):
    all_pseudo_tuned[:, j] = (probs_clip[:, j] > sed_best_thresholds[cls]).astype(int)

micro_tuned = f1_score(labels_all, all_pseudo_tuned, average="micro", zero_division=0)
macro_tuned = f1_score(labels_all, all_pseudo_tuned, average="macro", zero_division=0)
print(f"\nMicro-F1 flat  : {best_micro_f1:.4f}  (global T={best_threshold})")
print(f"Micro-F1 tuned : {micro_tuned:.4f}  (per-class T)")
print(f"Macro-F1 tuned : {macro_tuned:.4f}")
print(f"Improvement    : {micro_tuned - best_micro_f1:+.4f}")


## mAP + Bootstrap 95% CI
Final evaluation metrics for dissertation.

In [ ]:
# ── mAP + BOOTSTRAP 95% CI ───────────────────────────────────────
# Run on test set with best checkpoint
# Requires: test_probs (N, 8) float probabilities, test_labels (N, 8) binary

from sklearn.metrics import average_precision_score
import numpy as np

# ── Collect test set probabilities ───────────────────────────────
model.eval()
test_probs_list, test_labels_list = [], []

with torch.no_grad():
    for X, y in tqdm(test_loader, desc="Collecting test probs"):
        with torch.amp.autocast("cuda"):
            logits = model(X.to(DEVICE))
            if SED_MODE:
                probs = torch.sigmoid(logits.max(dim=1).values).float().cpu().numpy()
            else:
                probs = torch.sigmoid(logits).float().cpu().numpy()
        test_probs_list.append(probs)
        test_labels_list.append(y.numpy())

test_probs_raw  = np.vstack(test_probs_list)   # (N, 8) floats
test_labels_raw = np.vstack(test_labels_list)  # (N, 8) binary

# ── Per-class AP ─────────────────────────────────────────────────
print("=== PER-CLASS AVERAGE PRECISION ===\n")
print(f"{'Class':12s} {'AP':>8}")
print("-" * 22)
ap_scores = []
for j, cls in enumerate(CLASSES_LOCAL):
    if test_labels_raw[:, j].sum() == 0:
        print(f"{cls:12s} {'N/A':>8}  (no positives)")
        continue
    ap = average_precision_score(test_labels_raw[:, j], test_probs_raw[:, j])
    ap_scores.append(ap)
    print(f"{cls:12s} {ap:8.4f}")

macro_map = np.mean(ap_scores)
print(f"\nmAP (macro): {macro_map:.4f}")

# ── Bootstrap 95% CI ─────────────────────────────────────────────
print("\nRunning bootstrap (10,000 resamples)...")
N_BOOT = 10000
boot_maps = []

for _ in range(N_BOOT):
    idx  = np.random.choice(len(test_labels_raw), len(test_labels_raw), replace=True)
    aps  = []
    for j in range(len(CLASSES_LOCAL)):
        if test_labels_raw[idx, j].sum() == 0:
            continue
        aps.append(average_precision_score(
            test_labels_raw[idx, j], test_probs_raw[idx, j]
        ))
    boot_maps.append(np.mean(aps))

ci_low  = np.percentile(boot_maps, 2.5)
ci_high = np.percentile(boot_maps, 97.5)

print(f"\n=== FINAL EVALUATION RESULTS ===")
print(f"mAP (macro)   : {macro_map:.4f}")
print(f"95% CI        : [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Micro-F1      : {best_micro_f1:.4f}  (T={best_threshold})")
print(f"Micro-F1 tuned: {micro_tuned:.4f}  (per-class T)")
print(f"Macro-F1 tuned: {macro_tuned:.4f}")
print(f"\n→ Report as: mAP = {macro_map:.3f} (95% CI [{ci_low:.3f}, {ci_high:.3f}])")


## Improved CLAP Text Prompts
Contrastive prompts for weak classes (Crying, Shout, Siren).

In [ ]:
# ── LOAD CLAP MODEL (inserted — was missing from this notebook) ──
# model_clap was referenced below but never defined in this file;
# original code assumed it was already loaded in an earlier session.
# NOTE: confirm enable_fusion / checkpoint match whatever produced
# your original CLAP baseline numbers, or these prompts aren't
# comparable to that run.
import laion_clap

model_clap = laion_clap.CLAP_Module(enable_fusion=False)
model_clap.load_ckpt()   # downloads default pretrained checkpoint
model_clap.eval()
print("CLAP model loaded.")


In [ ]:
# ── IMPROVED TEXT PROMPTS (contrastive language) ─────────────────
# Rerun CLAP with tighter prompts for weak classes
# Requires CLAP model still loaded in memory

TEXT_PROMPTS_V2 = {
    "Scream":    [
        "a person screaming in fear or pain",
        "high pitched terrified scream",
        "screaming sound in a horror or action film",
        "someone screaming loudly in distress",
    ],
    "Shout":     [
        "person shouting aggressively, not screaming in fear",
        "loud commanding angry shout in an argument",
        "aggressive yelling, no crying or screaming",
        "person yelling commands or threats",
    ],
    "Crying":    [
        "person sobbing and weeping with tears, not screaming",
        "emotional crying sound only, no shouting or yelling",
        "quiet distressed sobbing, no loud screaming",
        "sound of someone crying with sniffling, no panic",
    ],
    "Explosion": [
        "loud explosion blast, sudden impact sound",
        "bomb or artillery explosion sound effect",
        "large explosion in action film, no music",
        "sudden explosive blast with reverb",
    ],
    "Gunshot":   [
        "single gunshot or gunfire sound",
        "sound of gun being fired, sharp crack",
        "multiple gunshots in rapid succession",
        "pistol or rifle shot sound effect",
    ],
    "Glass":     [
        "glass shattering and breaking sound",
        "window smashing, sound of broken glass",
        "glass object dropping and shattering",
        "sharp crashing sound of breaking glass only",
    ],
    "Siren":     [
        "emergency vehicle siren wailing, no music or speech",
        "high pitched ambulance siren sound only",
        "police car siren in distance, no background noise",
        "fire truck emergency siren, no voices",
    ],
    "Alarm":     [
        "alarm sound ringing loudly",
        "fire alarm or smoke detector beeping",
        "loud repeating alarm bell or buzzer",
        "emergency alarm sound, no speech or music",
    ],
}

# Generate new text embeddings
print("Generating improved text embeddings...")
text_embeddings_v2 = {}
with torch.no_grad():
    for cls, prompts in TEXT_PROMPTS_V2.items():
        embs = model_clap.get_text_embedding(prompts, use_tensor=True)
        avg  = embs.mean(dim=0)
        text_embeddings_v2[cls] = torch.nn.functional.normalize(avg, dim=0)
        print(f"  {cls} ✓")

print("\nText embeddings v2 ready.")
print("Run CLAP inference cell next with text_embeddings_v2")
print("Then compare F1 before/after prompt improvement")


## SED Error Analysis
Inspect worst FP/FN per class to understand remaining failures.

In [ ]:
# ── SED ERROR ANALYSIS — FALSE POSITIVES & NEGATIVES ────────────
# Uses: test_probs_raw (N,8), test_labels_raw (N,8), test_meta

print("=== TOP FALSE POSITIVES (predicted 1, truth 0) ===\n")

for j, cls in enumerate(CLASSES_LOCAL):
    t = sed_best_thresholds[cls]
    preds  = (test_probs_raw[:, j] > t).astype(int)
    fp_idx = np.where((preds == 1) & (test_labels_raw[:, j] == 0))[0]
    fp_sorted = fp_idx[np.argsort(-test_probs_raw[fp_idx, j])][:3]

    true_classes_list = []
    for idx in fp_sorted:
        true_cls = [CLASSES_LOCAL[k] for k in range(8) if test_labels_raw[idx, k] == 1]
        true_classes_list.append(true_cls or ["negative"])

    print(f"{cls:12s} — {len(fp_idx)} FPs total")
    for i, idx in enumerate(fp_sorted):
        fname = test_meta.iloc[idx]["fname"] if hasattr(test_meta, 'iloc') else idx
        print(f"  score={test_probs_raw[idx, j]:.3f}  true={true_classes_list[i]}")
    print()

print("=== TOP FALSE NEGATIVES (predicted 0, truth 1) ===\n")

for j, cls in enumerate(CLASSES_LOCAL):
    t = sed_best_thresholds[cls]
    preds  = (test_probs_raw[:, j] > t).astype(int)
    fn_idx = np.where((preds == 0) & (test_labels_raw[:, j] == 1))[0]
    fn_sorted = fn_idx[np.argsort(test_probs_raw[fn_idx, j])][:3]

    print(f"{cls:12s} — {len(fn_idx)} FNs total | {len(fn_idx)}/{test_labels_raw[:,j].sum():.0f} missed")
    for idx in fn_sorted:
        print(f"  score={test_probs_raw[idx, j]:.3f}")
    print()


## SED Inference — Event Detection on a Single Clip